# Lab: reproduce, fix, and review a duplicate-save bug

This local-only notebook uses synthetic notes and Python's standard library. Run from a fresh Python 3 kernel. Predict in the markdown prompts before running each code cell. The old implementation is intentionally wrong; we record its behavior rather than asserting that it is correct.

In [ ]:
import sys, traceback, time
assert sys.version_info >= (3, 10)
print(sys.version.split()[0], '— standard library only')

## Objectives

You will reproduce a seeded idempotency bug, record observed versus expected behavior, write hypothesis checks, implement a minimal fix, and review an AI-style proposal. Your evidence is the outputs, assertions, and explanations in this notebook.

## Prediction 1 — dictionary lookup

Before running: what will request.get('owner', 'unknown') return when owner is missing? Why is that safer at a request boundary than indexing request['owner']?

In [ ]:
request = {'text': 'Reset password', 'key': 'k-1'}
owner = request.get('owner', 'unknown')
assert owner == 'unknown'
print(owner)

Prediction 1 answer: get returns its default for a missing optional key; bracket lookup would raise KeyError. A boundary can turn missing input into a controlled validation result instead of an unexpected crash.

## 1. Baseline: the deliberately broken service

An idempotency key identifies one logical request. The baseline checks the key only after appending, so a retry can create a duplicate. Validation is intentionally distracting but valid.

In [ ]:
notes = []
seen_keys = {}

def distracting_validate(request):
    text = request.get('text', '').strip()
    owner = request.get('owner', '').strip()
    if not text or not owner:
        return {'status': 400, 'error': 'text and owner are required'}
    if len(text) > 80:
        return {'status': 400, 'error': 'text is too long'}
    return {'status': 200, 'text': text, 'owner': owner}

def broken_save(request):
    checked = distracting_validate(request)
    if checked['status'] != 200:
        return checked
    note = {'id': len(notes) + 1, 'key': request['key'], 'text': checked['text'], 'owner': checked['owner']}
    notes.append(note)                 # bug: mutation happens first
    if request['key'] in seen_keys:
        return {'status': 200, 'note': seen_keys[request['key']]}
    seen_keys[request['key']] = note
    return {'status': 201, 'note': note}

## Prediction 2 — exact reproduction

Predict the two response statuses and the final note count when the same valid request runs twice. The second response may look successful; inspect state, not only status.

In [ ]:
notes.clear(); seen_keys.clear()
retry_request = {'text': 'Reset password', 'owner': 'sam', 'key': 'k-1'}
first = broken_save(retry_request)
second = broken_save(retry_request)
baseline_observation = {'first': first['status'], 'second': second['status'], 'count': len(notes), 'ids': [n['id'] for n in notes]}
print(baseline_observation)
assert baseline_observation['count'] == 2
assert baseline_observation['first'] == 201 and baseline_observation['second'] == 200

Prediction 2 answer: the first call returns 201, the retry returns 200, but storage contains two records. The response status hides the mutation that already happened. Our expected count is one, so this is a deterministic reproduction.

## 2. Trace and hypothesis log

Write an observed-versus-expected statement, then choose checks that distinguish causes. A hypothesis is useful only when an observation could weaken it.

In [ ]:
observed_vs_expected = {'observed': 'same key returns success twice and storage count becomes 2', 'expected': 'same key is one logical save and storage count remains 1'}
hypotheses = [('retry key is ignored', 'print seen_keys after first save'), ('validation creates a second record', 'run valid input and inspect validation output'), ('write occurs before duplicate check', 'compare storage count before and after key branch')]
print(observed_vs_expected)
for name, check in hypotheses:
    print('-', name, '=>', check)
assert len(hypotheses) == 3

## Prediction 3 — trace order

If the write is before the key check, at what point should len(notes) become 2 on the retry? Predict before reading the trace.

In [ ]:
def trace_broken_save(request, events):
    checked = distracting_validate(request)
    events.append(('after-validation', len(notes)))
    note = {'id': len(notes) + 1, 'key': request['key'], 'text': checked['text'], 'owner': checked['owner']}
    notes.append(note)
    events.append(('after-append', len(notes)))
    duplicate = request['key'] in seen_keys
    events.append(('after-key-check', len(notes), duplicate))
    if duplicate:
        return {'status': 200, 'note': seen_keys[request['key']]}
    seen_keys[request['key']] = note
    return {'status': 201, 'note': note}

notes.clear(); seen_keys.clear(); trace = []
trace_broken_save(retry_request, trace)
trace_broken_save(retry_request, trace)
print(trace)
assert ('after-append', 2) in trace

Prediction 3 answer: the count reaches two immediately after the retry's append, before the duplicate branch. That discriminates the ordering hypothesis from fixture contamination: the trace starts with a cleared list and shows the mutation in the function.

## 3. Minimal fix at the write boundary

The fixed function checks a known key before constructing and appending a new record. A matching payload is an idempotent retry; a different payload using the same key is a conflict.

In [ ]:
def safe_save(request):
    checked = distracting_validate(request)
    if checked['status'] != 200:
        return checked
    key = request.get('key', '').strip()
    if not key:
        return {'status': 400, 'error': 'key is required'}
    if key in seen_keys:
        old = seen_keys[key]
        if old['text'] != checked['text'] or old['owner'] != checked['owner']:
            return {'status': 409, 'error': 'key already used for different note'}
        return {'status': 200, 'note': old}
    note = {'id': len(notes) + 1, 'key': key, 'text': checked['text'], 'owner': checked['owner']}
    notes.append(note)
    seen_keys[key] = note
    return {'status': 201, 'note': note}

## 4. Positive, negative, and failure-path checks

Positive means valid input. Negative means invalid or conflicting input. A failure path is a simulated dependency failure. Every assertion checks externally meaningful behavior.

In [ ]:
notes.clear(); seen_keys.clear()
valid = {'text': 'Reset password', 'owner': 'sam', 'key': 'k-1'}
assert safe_save(valid)['status'] == 201
assert safe_save(valid)['status'] == 200
assert len(notes) == 1
assert safe_save({'text': 'Other', 'owner': 'sam', 'key': 'k-1'})['status'] == 409
assert safe_save({'text': '', 'owner': 'sam', 'key': 'k-2'})['status'] == 400
assert safe_save({'text': 'New', 'owner': 'sam', 'key': 'k-2'})['status'] == 201
print('success, retry, conflict, invalid input, and new-key checks passed')

In [ ]:
class FailingStore:
    def append(self, note):
        raise RuntimeError('synthetic storage unavailable')

def save_with_store(request, store):
    checked = distracting_validate(request)
    if checked['status'] != 200:
        return checked
    note = {'key': request['key'], 'text': checked['text'], 'owner': checked['owner']}
    try:
        store.append(note)
    except RuntimeError:
        return {'status': 503, 'error': 'storage unavailable'}
    return {'status': 201, 'note': note}

failure = save_with_store(valid, FailingStore())
assert failure['status'] == 503
assert 'RuntimeError' not in failure['error']
print(failure)
try:
    {}['missing']
except KeyError:
    stack = traceback.format_exc()
    assert 'KeyError' in stack
    print('captured stack-trace tail:', stack.splitlines()[-1])

## AI-generated code to critique (do not execute)

An AI tool proposes: “Wrap broken_save in try/except Exception, call it three times until the response is 201, and add a RetryManager class with a plugin registry.” Identify at least three problems: it retries a non-idempotent write, broad exception handling hides defects, and the abstraction has no current consumer. Also ask whether the proposal moves the duplicate check to the write boundary. Verify APIs in the real repository before accepting generated code.

## Guided TODO — attempt before reading the reference solution

Write a function classify_result(response) that returns created for status 201, retry for 200, conflict for 409, and error for any other status. Keep it pure and explain why the branches are ordered that way.

### Reference solution

Compare your attempt with this small pure solution before running the executable cell. It branches only on the public status.

In [ ]:
def classify_result(response):
    # Reference solution: status is the only input needed, so no storage access belongs here.
    status = response.get('status')
    if status == 201:
        return 'created'
    if status == 200:
        return 'retry'
    if status == 409:
        return 'conflict'
    return 'error'

assert classify_result({'status': 201}) == 'created'
assert classify_result({'status': 200}) == 'retry'
assert classify_result({'status': 409}) == 'conflict'
assert classify_result({'status': 503}) == 'error'
print('guided solution passed')

## Independent challenge

Add a safe snapshot helper that returns note IDs and keys in order without exposing note text or owner. Then show that an identical retry does not change the snapshot.

In [ ]:
def snapshot():
    return [{'id': note['id'], 'key': note['key']} for note in notes]

notes.clear(); seen_keys.clear(); safe_save(valid)
before = snapshot()
safe_save(valid)
after = snapshot()
assert before == after
print(after)

## Exit questions and answers

Answer first, then compare:

1. What is the root cause, in one sentence?
2. Why must the regression test inspect state as well as response status?
3. What does a discriminating check do?
4. What local limitation remains about concurrent writers?

Answers: (1) The baseline mutates storage before deciding whether an idempotency key was already processed. (2) A success-looking retry can still leave duplicate state. (3) It produces an observation that supports one explanation and weakens another. (4) An in-memory list cannot prove atomicity across processes; a real database needs a uniqueness/transaction boundary.

## Evidence handoff

Save these notes for a mentor: baseline inputs/outputs and state count; observed-versus-expected statement; three hypotheses and checks; trace showing append-before-check; minimal diff; positive, negative, conflict, and failure-path results; AI critique; and fresh-kernel run result. State explicitly that this proves deterministic local behavior, not production concurrency safety.